# Dataset sample for slides

One random image per class, in 1 row of 10. Neighbouring images form one confusable pair, so the slide also shows why the task is hard.

Images come from the training data only, chosen by seed rather than by hand. They are centre-cropped to squares for display only.


In [ ]:
VARIANT = "original"  # original, clahe, histeq, gsam, gsam_crop, gsam_light, gsam_light_crop, sam, sam_crop
SEED = 0              # change to see other random examples
SAVE_FIGURE = True    # writes a PNG to analyst/figures/


In [ ]:
import os
import random
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image, ImageOps

# ---- paths (preset) -------------------------------------------------------
# project root = the folder that contains pipeline_preprocess/ (works from any sub-folder)
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pipeline_preprocess").is_dir())
DATA_DIR = Path(os.environ.get("FOOD_DATA_DIR", PROJECT_ROOT / "data"))
FIGURE_DIR = PROJECT_ROOT / "analyst" / "figures"

# same position in both lists = one pair, drawn side by side
FIRST_OF_PAIR = ["ramen", "steak", "fried_rice", "chocolate_cake", "caesar_salad"]
SECOND_OF_PAIR = ["pho", "filet_mignon", "pad_thai", "chocolate_mousse", "greek_salad"]


In [ ]:
variant_dir = DATA_DIR / "preprocessing_data" / f"{VARIANT}_data"
rng = random.Random(SEED)

def class_paths(cls):
    if variant_dir.is_dir():
        return sorted((variant_dir / "train" / cls).glob("*.jpg"))
    # generated data not here yet: use the official train list so test images stay unseen
    assert VARIANT == "original", f"{variant_dir} not found. Unzip it from data_zips/ on Drive."
    raw = DATA_DIR / "raw_data"
    lines = open(raw / "meta" / "meta" / "train.txt").read().split()
    return [raw / "images" / f"{line}.jpg" for line in lines if line.split("/")[0] == cls]


In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(20, 3))

for slots, classes in zip((axes[0::2], axes[1::2]), (FIRST_OF_PAIR, SECOND_OF_PAIR)):
    for ax, cls in zip(slots, classes):
        path = rng.choice(class_paths(cls))
        ax.imshow(ImageOps.fit(Image.open(path).convert("RGB"), (384, 384)))
        ax.set_title(cls.replace("_", " "), fontsize=12)
        ax.axis("off")

fig.suptitle("Food Images (Food-101) Version 5", fontsize=18, fontweight="bold")
fig.tight_layout()

if SAVE_FIGURE:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURE_DIR / f"dataset_sample_{VARIANT}_seed{SEED}.png", dpi=200, bbox_inches="tight")
plt.show()
